# Medicare Opioid Prescribing Trends

*Normalized Analysis of High-Risk Prescribing Across Provider Types*

This project analyzes Medicare Part D prescribing data from 2017-2023 to identify high-risk opioid prescribing patterns across provider types, regions, and rural/urban areas. Normalized rates, calculated as opioid claims relative to total claims, reveal meaningful differences that raw claim counts can hide. Long-acting opioids and overall opioid use are examined alongside buprenorphine for opioid use disorder (OUD) treatment to separate pain-management interpretation from treatment-access interpretation. Machine learning models are used as screening tools to identify providers with elevated prescribing risk, not to prove clinical wrongdoing.

The notebook supports the same core story used in `Presentation9_Refined_Template_Preserved.pdf` and `Project2Poster_Refined.png`: normalized methodology, provider-type differences, rural/urban comparison, long-acting opioid share, OUD versus pain-management distinction, model-based screening, limitations, and practical recommendations.

## Root-Cause Analysis

Schema validation was necessary before analysis because the CMS files do not use one consistent opioid column naming pattern. The provider summary file uses columns such as `Opioid_Tot_Clms` and `Opioid_LA_Tot_Clms`, while the geography opioid-rate file uses `Tot_Opioid_Clms` and `LA_Tot_Opioid_Clms`. Using the wrong column name caused earlier missing-column failures and would have produced incomplete results. Checking schemas before analysis prevents silent mistakes and makes the notebook safer to rerun from start to finish.

A second root cause was overly strict credential filtering. Exact-match mapping dropped valid credentials such as `MD, MPH`, `NP-C`, `DNP, FNP-C`, and `PA-C`. The corrected classifier searches within the credential text and keeps providers whose credentials clearly match MD, DO, PA, or NP categories. This keeps the analysis aligned with the intended provider population.

Data and methods match the presentation scope: CMS Medicare Part D prescribing data from 2017-2023; providers limited to MD, DO, NP, and PA credentials; key measures including normalized opioid rate, long-acting opioid share, and buprenorphine/OUD separation; analysis by provider type, specialty, region, rural/urban status, and Washington versus U.S. regions; and model comparison across Logistic Regression, Decision Tree, and Random Forest classifiers.


In [ ]:
# ============================================
# 00 - PROJECT SETUP AND DATA VALIDATION
# ============================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", palette="colorblind")

PROJECT_DIR = Path.cwd()
DATA_FILES = {
    "provider": PROJECT_DIR / "Medicare_Part_D_Prescribers_by_Provider_2023.csv",
    "geo_drug": PROJECT_DIR / "Medicare_Part_D_Prescribers_by_Geography_and_Drug_2023.csv",
    "medicare_geo": PROJECT_DIR / "Medicare Part D Opioid Prescribing Rates - by Geography" / "2023" / "OMT_MDCR_RY25_P04_V10_Y23_GEO.csv",
    "medicaid_geo": PROJECT_DIR / "Medicaid Opioid Prescribing Rates - by Geography" / "2023" / "OMT_MDCD_RY25_P02_V10_YTD23_GEO.csv",
}

missing_files = [name for name, path in DATA_FILES.items() if not path.exists()]
if missing_files:
    missing_display = "\n".join(f"- {name}: {DATA_FILES[name]}" for name in missing_files)
    raise FileNotFoundError(f"Missing required data files:\n{missing_display}")

PROVIDER_REQUIRED = [
    "Prscrbr_Crdntls", "Prscrbr_State_Abrvtn", "Prscrbr_RUCA", "Prscrbr_Type",
    "Tot_Clms", "Opioid_Tot_Clms", "Opioid_Prscrbr_Rate", "Opioid_LA_Tot_Clms",
]
GEOGRAPHY_REQUIRED = [
    "Year", "Prscrbr_Geo_Lvl", "Prscrbr_Geo_Desc", "Breakout_Type", "Breakout",
    "Tot_Opioid_Clms", "Tot_Clms", "Opioid_Prscrbng_Rate", "LA_Tot_Opioid_Clms", "LA_Opioid_Prscrbng_Rate",
]
GEO_DRUG_REQUIRED = ["Prscrbr_Geo_Lvl", "Prscrbr_Geo_Desc", "Brnd_Name", "Gnrc_Name", "Tot_Clms", "Opioid_Drug_Flag", "Opioid_LA_Drug_Flag"]

provider_columns = pd.read_csv(DATA_FILES["provider"], nrows=0).columns.tolist()
medicare_geo_columns = pd.read_csv(DATA_FILES["medicare_geo"], nrows=0).columns.tolist()
geo_drug_columns = pd.read_csv(DATA_FILES["geo_drug"], nrows=0).columns.tolist()

def require_columns(available, required, dataset_name):
    missing = [col for col in required if col not in available]
    if missing:
        raise KeyError(f"{dataset_name} is missing required columns: {missing}")

require_columns(provider_columns, PROVIDER_REQUIRED, "Provider dataset")
require_columns(medicare_geo_columns, GEOGRAPHY_REQUIRED, "Medicare geography dataset")
require_columns(geo_drug_columns, GEO_DRUG_REQUIRED, "Geography and drug dataset")

print("Project directory:", PROJECT_DIR)
print("Provider columns:", len(provider_columns))
print("Medicare geography columns:", len(medicare_geo_columns))
print("Geography + drug columns:", len(geo_drug_columns))

## 01 - Consensus Data Pipeline

The cleaning pipeline prepares a reliable provider-level dataset for analysis. It checks required columns, removes duplicate rows, filters only truly negative numeric values, applies the `Tot_Clms >= 11` threshold, standardizes credentials, removes invalid states and territories, maps regions, and creates rural/urban and normalized opioid variables. These steps reduce data-quality problems without removing legitimate CMS suppressed or missing values. The flexible credential classifier also prevents valid providers such as `MD, MPH`, `NP-C`, `DNP, FNP-C`, and `PA-C` from being incorrectly dropped. A clean and well-documented pipeline makes the downstream analysis more reliable and easier to explain in the report.


In [ ]:
# ============================================
# 01 - DATA CLEANING
# ============================================

STATE_ABBR_TO_NAME = {
    "AL": "Alabama", "AK": "Alaska", "AZ": "Arizona", "AR": "Arkansas", "CA": "California", "CO": "Colorado",
    "CT": "Connecticut", "DE": "Delaware", "FL": "Florida", "GA": "Georgia", "HI": "Hawaii", "ID": "Idaho",
    "IL": "Illinois", "IN": "Indiana", "IA": "Iowa", "KS": "Kansas", "KY": "Kentucky", "LA": "Louisiana",
    "ME": "Maine", "MD": "Maryland", "MA": "Massachusetts", "MI": "Michigan", "MN": "Minnesota", "MS": "Mississippi",
    "MO": "Missouri", "MT": "Montana", "NE": "Nebraska", "NV": "Nevada", "NH": "New Hampshire", "NJ": "New Jersey",
    "NM": "New Mexico", "NY": "New York", "NC": "North Carolina", "ND": "North Dakota", "OH": "Ohio", "OK": "Oklahoma",
    "OR": "Oregon", "PA": "Pennsylvania", "RI": "Rhode Island", "SC": "South Carolina", "SD": "South Dakota", "TN": "Tennessee",
    "TX": "Texas", "UT": "Utah", "VT": "Vermont", "VA": "Virginia", "WA": "Washington", "WV": "West Virginia",
    "WI": "Wisconsin", "WY": "Wyoming",
}
STATE_NAME_TO_ABBR = {name: abbr for abbr, name in STATE_ABBR_TO_NAME.items()}

REGION_MAP = {
    "ME": "Northeast", "NH": "Northeast", "VT": "Northeast", "MA": "Northeast", "RI": "Northeast", "CT": "Northeast", "NY": "Northeast", "NJ": "Northeast", "PA": "Northeast",
    "OH": "Midwest", "MI": "Midwest", "IN": "Midwest", "IL": "Midwest", "WI": "Midwest", "MN": "Midwest", "IA": "Midwest", "MO": "Midwest", "ND": "Midwest", "SD": "Midwest", "NE": "Midwest", "KS": "Midwest",
    "DE": "South", "MD": "South", "VA": "South", "WV": "South", "KY": "South", "TN": "South", "NC": "South", "SC": "South", "GA": "South", "FL": "South", "AL": "South", "MS": "South", "AR": "South", "LA": "South",
    "AZ": "Southwest", "NM": "Southwest", "OK": "Southwest", "TX": "Southwest",
    "CO": "West", "WY": "West", "MT": "West", "ID": "West", "UT": "West", "NV": "West", "CA": "West", "OR": "West", "WA": "West", "AK": "West", "HI": "West",
}

provider_df = pd.read_csv(DATA_FILES["provider"], low_memory=False)
original_rows = len(provider_df)

duplicate_rows = provider_df.duplicated().sum()
provider_df = provider_df.drop_duplicates().copy()

numeric_columns = provider_df.select_dtypes(include=["number"]).columns.tolist()
negative_counts = (provider_df[numeric_columns] < 0).sum().sort_values(ascending=False)
negative_counts = negative_counts[negative_counts > 0]
provider_df = provider_df[~(provider_df[numeric_columns] < 0).any(axis=1)].copy()

provider_df = provider_df[pd.to_numeric(provider_df["Tot_Clms"], errors="coerce") >= 11].copy()

# ============================================
# FLEXIBLE CREDENTIAL CLEANING
# ============================================

provider_df["credential_text"] = (
    provider_df["Prscrbr_Crdntls"]
    .fillna("")
    .astype(str)
    .str.upper()
)

def classify_credential(text):
    if any(x in text for x in ["MD", "M.D"]):
        return "MD"
    elif any(x in text for x in ["DO", "D.O"]):
        return "DO"
    elif any(x in text for x in ["PA", "PA-C", "PHYSICIAN ASSISTANT"]):
        return "PA"
    elif any(x in text for x in ["NP", "FNP", "DNP", "ARNP", "APRN", "NURSE PRACTITIONER"]):
        return "NP"
    else:
        return np.nan

provider_df["credential_clean"] = provider_df["credential_text"].apply(classify_credential)

# ============================================
# KEEP TARGET PROVIDERS
# ============================================

provider_df = provider_df[provider_df["credential_clean"].isin(["MD", "DO", "PA", "NP"])].copy()

print(provider_df["credential_clean"].value_counts())
print("\nRemaining rows:")
print(len(provider_df))

provider_df["Prscrbr_State_Abrvtn"] = provider_df["Prscrbr_State_Abrvtn"].astype(str).str.upper().str.strip()
provider_df = provider_df[provider_df["Prscrbr_State_Abrvtn"].isin(STATE_ABBR_TO_NAME)].copy()
provider_df["state_name"] = provider_df["Prscrbr_State_Abrvtn"].map(STATE_ABBR_TO_NAME)
provider_df["region"] = provider_df["Prscrbr_State_Abrvtn"].map(REGION_MAP)

provider_df["Prscrbr_RUCA"] = pd.to_numeric(provider_df["Prscrbr_RUCA"], errors="coerce")
provider_df["area_type"] = np.where(provider_df["Prscrbr_RUCA"] <= 3, "Urban", "Rural")
provider_df.loc[provider_df["Prscrbr_RUCA"].isna(), "area_type"] = np.nan

for col in ["Tot_Clms", "Opioid_Tot_Clms", "Opioid_LA_Tot_Clms", "Tot_Benes", "Tot_Drug_Cst", "Bene_Avg_Age", "Bene_Avg_Risk_Scre"]:
    provider_df[col] = pd.to_numeric(provider_df[col], errors="coerce")

provider_df["Opioid_Tot_Clms"] = provider_df["Opioid_Tot_Clms"].fillna(0)
provider_df["Opioid_LA_Tot_Clms"] = provider_df["Opioid_LA_Tot_Clms"].fillna(0)
provider_df["opioid_rate_normalized"] = np.where(provider_df["Tot_Clms"] > 0, provider_df["Opioid_Tot_Clms"] / provider_df["Tot_Clms"], np.nan)
provider_df["la_opioid_share"] = np.where(provider_df["Opioid_Tot_Clms"] > 0, provider_df["Opioid_LA_Tot_Clms"] / provider_df["Opioid_Tot_Clms"], 0)

high_risk_threshold = provider_df.loc[provider_df["Opioid_Tot_Clms"] >= 11, "opioid_rate_normalized"].quantile(0.90)
provider_df["high_risk"] = (
    (provider_df["Opioid_Tot_Clms"] >= 11) &
    (provider_df["opioid_rate_normalized"] >= high_risk_threshold)
).astype(int)

clean_provider_df = provider_df.copy()

cleaning_summary = pd.DataFrame({
    "step": ["original_rows", "duplicates_removed", "rows_after_all_filters", "high_risk_threshold"],
    "value": [original_rows, int(duplicate_rows), len(provider_df), round(float(high_risk_threshold), 4)],
})
print(cleaning_summary)
print("\nNegative value counts removed:")
print(negative_counts if len(negative_counts) else "No negative numeric values found.")
print("\nMissing values in key fields:")
print(provider_df[["credential_clean", "Prscrbr_State_Abrvtn", "region", "area_type", "opioid_rate_normalized"]].isna().sum())
print("\nCleaned provider dataframe is available as clean_provider_df.")


## 02 - Medicare Geography Trends: Washington vs Regions, 2017-2023

Washington is compared with broader regions to show whether its opioid prescribing pattern differs from national regional trends. The Medicare geography file includes explicit year-by-year data for each year from 2017 through 2023. The analysis calculates each year separately before comparing Washington with regional weighted rates.

The analysis uses weighted opioid prescribing rates, calculated as opioid claims relative to total provider claims and aggregated longitudinally across 2017-2023. Weighted rates are better than raw claim totals because they account for total prescribing volume, so larger and smaller states can be compared more fairly. Washington's Medicare Part D opioid prescribing rate declined steadily from 2017 to 2023, but in 2023 it remained higher than the aggregated regional weighted rates shown in this analysis. This corrected wording matches the data table more closely than the older poster wording. The Washington-versus-region comparison remains useful because it shows both longitudinal improvement and the importance of aggregation methodology.


In [ ]:
# ============================================
# 02 - GEOGRAPHY TRENDS AND WASHINGTON COMPARISON
# ============================================

medicare_geo_df = pd.read_csv(DATA_FILES["medicare_geo"], low_memory=False)
medicare_geo_df = medicare_geo_df[(medicare_geo_df["Year"].between(2017, 2023))].copy()

state_geo = medicare_geo_df[
    (medicare_geo_df["Prscrbr_Geo_Lvl"].eq("State")) &
    (medicare_geo_df["Breakout_Type"].eq("Totals")) &
    (medicare_geo_df["Breakout"].eq("Overall")) &
    (medicare_geo_df["Prscrbr_Geo_Desc"].isin(STATE_NAME_TO_ABBR))
].copy()
state_geo["state_abbr"] = state_geo["Prscrbr_Geo_Desc"].map(STATE_NAME_TO_ABBR)
state_geo["region"] = state_geo["state_abbr"].map(REGION_MAP)
state_geo["weighted_opioid_rate"] = np.where(state_geo["Tot_Clms"] > 0, state_geo["Tot_Opioid_Clms"] / state_geo["Tot_Clms"] * 100, np.nan)
state_geo["weighted_la_rate"] = np.where(state_geo["Tot_Clms"] > 0, state_geo["LA_Tot_Opioid_Clms"] / state_geo["Tot_Clms"] * 100, np.nan)

region_trends = (
    state_geo.groupby(["Year", "region"], as_index=False)
    .agg(Tot_Opioid_Clms=("Tot_Opioid_Clms", "sum"), Tot_Clms=("Tot_Clms", "sum"), LA_Tot_Opioid_Clms=("LA_Tot_Opioid_Clms", "sum"))
)
region_trends["weighted_opioid_rate"] = region_trends["Tot_Opioid_Clms"] / region_trends["Tot_Clms"] * 100
region_trends["weighted_la_rate"] = region_trends["LA_Tot_Opioid_Clms"] / region_trends["Tot_Clms"] * 100

washington_trend = state_geo[state_geo["state_abbr"].eq("WA")].copy()
washington_vs_regions = pd.concat([
    region_trends.assign(series=region_trends["region"]),
    washington_trend.assign(series="Washington")[["Year", "series", "Tot_Opioid_Clms", "Tot_Clms", "LA_Tot_Opioid_Clms", "weighted_opioid_rate", "weighted_la_rate"]],
], ignore_index=True)

plt.figure(figsize=(11, 6))
sns.lineplot(data=washington_vs_regions, x="Year", y="weighted_opioid_rate", hue="series", marker="o")
plt.title("Medicare Part D Opioid Prescribing Rate: Washington vs Regions, 2017-2023")
plt.ylabel("Opioid claims per 100 total claims")
plt.xlabel("Year")
plt.tight_layout()
plt.show()

rural_urban_geo = medicare_geo_df[
    (medicare_geo_df["Prscrbr_Geo_Lvl"].eq("National")) &
    (medicare_geo_df["Breakout_Type"].eq("Rural/Urban"))
].copy()
rural_urban_geo["weighted_opioid_rate"] = rural_urban_geo["Tot_Opioid_Clms"] / rural_urban_geo["Tot_Clms"] * 100

plt.figure(figsize=(9, 5))
sns.lineplot(data=rural_urban_geo, x="Year", y="weighted_opioid_rate", hue="Breakout", marker="o")
plt.title("National Rural vs Urban Opioid Prescribing Rate, 2017-2023")
plt.ylabel("Opioid claims per 100 total claims")
plt.xlabel("Year")
plt.tight_layout()
plt.show()

print("Washington trend:")
print(washington_trend[["Year", "Tot_Opioid_Clms", "Tot_Clms", "weighted_opioid_rate", "weighted_la_rate"]])
print("\nLatest regional comparison:")
print(region_trends[region_trends["Year"].eq(2023)].sort_values("weighted_opioid_rate", ascending=False))

yearly_comparison = (
    washington_vs_regions
    .pivot_table(index="Year", columns="series", values="weighted_opioid_rate")
    .round(3)
)
print("\nExplicit year-by-year weighted opioid rate comparison, 2017-2023:")
print(yearly_comparison)


## 03 - EDA, Normalized Analysis, Long-Acting Opioids, and Buprenorphine/OUD

Provider-level results should be interpreted using normalized rates, not only raw opioid claim counts. The analysis compares credentials, specialties, regions, rural/urban groups, Washington-only providers, long-acting opioid share, and outliers using `opioid_rate_normalized` and related measures. Normalized rates show prescribing intensity, while raw counts mostly show provider volume. This supports the poster's provider-type conclusion: Nurse Practitioners and Physician Assistants show higher normalized opioid prescribing rates on average than MDs and DOs.

The rural/urban analysis supports the presentation finding that rural providers have higher normalized opioid prescribing rates than urban providers across the analyzed years. This matters because rural prescribing patterns may reflect provider access, patient mix, specialty distribution, and regional healthcare constraints rather than provider behavior alone.

Long-acting opioid measures add a high-risk prescribing lens because they capture a more intensive opioid category. Long-acting opioids represent a smaller share of prescriptions overall, but high long-acting opioid shares concentrate among a smaller group of providers. These comparisons help identify which provider groups and settings may need closer review or targeted intervention.

Buprenorphine and OUD-related medications should not be interpreted the same way as pain-management opioids. The drug-level analysis separates OUD-related buprenorphine products from pain-management opioid products. High buprenorphine prescribing can indicate better access to opioid use disorder treatment, not necessarily unsafe prescribing. This distinction prevents misleading conclusions and should be discussed in the report as a treatment-access signal as well as an opioid-related measure.


In [ ]:
# ============================================
# 03 - EXPLORATORY AND ADVANCED ANALYSIS
# ============================================

df = clean_provider_df.copy()

provider_summary = df.groupby("credential_clean", as_index=False).agg(
    providers=("PRSCRBR_NPI", "nunique"),
    total_claims=("Tot_Clms", "sum"),
    opioid_claims=("Opioid_Tot_Clms", "sum"),
    la_opioid_claims=("Opioid_LA_Tot_Clms", "sum"),
    mean_normalized_rate=("opioid_rate_normalized", "mean"),
    median_normalized_rate=("opioid_rate_normalized", "median"),
    mean_la_share=("la_opioid_share", "mean"),
    high_risk_rate=("high_risk", "mean"),
).sort_values("mean_normalized_rate", ascending=False)

region_summary = df.groupby("region", as_index=False).agg(
    providers=("PRSCRBR_NPI", "nunique"),
    opioid_claims=("Opioid_Tot_Clms", "sum"),
    total_claims=("Tot_Clms", "sum"),
    mean_normalized_rate=("opioid_rate_normalized", "mean"),
    high_risk_rate=("high_risk", "mean"),
)
region_summary["weighted_opioid_rate"] = region_summary["opioid_claims"] / region_summary["total_claims"]

area_summary = df.groupby("area_type", as_index=False).agg(
    providers=("PRSCRBR_NPI", "nunique"),
    mean_normalized_rate=("opioid_rate_normalized", "mean"),
    mean_la_share=("la_opioid_share", "mean"),
    high_risk_rate=("high_risk", "mean"),
)

specialty_summary = (
    df.groupby("Prscrbr_Type", as_index=False)
    .agg(providers=("PRSCRBR_NPI", "nunique"), mean_normalized_rate=("opioid_rate_normalized", "mean"), opioid_claims=("Opioid_Tot_Clms", "sum"), high_risk_rate=("high_risk", "mean"))
    .query("providers >= 100")
    .sort_values("mean_normalized_rate", ascending=False)
    .head(15)
)

wa_provider_summary = df[df["Prscrbr_State_Abrvtn"].eq("WA")].groupby("credential_clean", as_index=False).agg(
    providers=("PRSCRBR_NPI", "nunique"),
    mean_normalized_rate=("opioid_rate_normalized", "mean"),
    opioid_claims=("Opioid_Tot_Clms", "sum"),
    high_risk_rate=("high_risk", "mean"),
).sort_values("mean_normalized_rate", ascending=False)

highest_risk_group = provider_summary.iloc[0]
print("Provider summary:")
print(provider_summary)
print("\nRural/urban summary:")
print(area_summary)
print("\nTop specialties by normalized opioid rate:")
print(specialty_summary)
print("\nWashington-only provider summary:")
print(wa_provider_summary)
print("\nHighest-risk provider credential group by mean normalized rate:")
print(highest_risk_group[["credential_clean", "mean_normalized_rate", "high_risk_rate"]])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.barplot(data=provider_summary, x="credential_clean", y="mean_normalized_rate", ax=axes[0, 0])
axes[0, 0].set_title("Normalized Opioid Rate by Provider Credential")
axes[0, 0].set_xlabel("Credential")
axes[0, 0].set_ylabel("Mean opioid claims / total claims")

sns.barplot(data=region_summary.sort_values("weighted_opioid_rate", ascending=False), x="region", y="weighted_opioid_rate", ax=axes[0, 1])
axes[0, 1].set_title("Weighted Opioid Rate by Region")
axes[0, 1].set_xlabel("Region")
axes[0, 1].set_ylabel("Weighted opioid claims / total claims")
axes[0, 1].tick_params(axis="x", rotation=25)

sns.barplot(data=area_summary, x="area_type", y="mean_normalized_rate", ax=axes[1, 0])
axes[1, 0].set_title("Rural vs Urban Normalized Rate")
axes[1, 0].set_xlabel("Area")
axes[1, 0].set_ylabel("Mean opioid claims / total claims")

sns.barplot(data=wa_provider_summary, x="credential_clean", y="mean_normalized_rate", ax=axes[1, 1])
axes[1, 1].set_title("Washington Providers by Credential")
axes[1, 1].set_xlabel("Credential")
axes[1, 1].set_ylabel("Mean opioid claims / total claims")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(df["opioid_rate_normalized"].dropna().clip(upper=df["opioid_rate_normalized"].quantile(0.99)), bins=40)
plt.title("Distribution of Normalized Opioid Prescribing Rates")
plt.xlabel("Opioid claims / total claims, clipped at 99th percentile")
plt.tight_layout()
plt.show()

corr_cols = ["Tot_Clms", "Tot_Benes", "Tot_Drug_Cst", "Bene_Avg_Age", "Bene_Avg_Risk_Scre", "Opioid_Tot_Clms", "Opioid_LA_Tot_Clms", "opioid_rate_normalized", "la_opioid_share"]
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation Matrix for Provider-Level Opioid Measures")
plt.tight_layout()
plt.show()

outlier_cutoff = df["opioid_rate_normalized"].quantile(0.99)
outliers = df[df["opioid_rate_normalized"] >= outlier_cutoff].copy()
print(f"\nOutlier threshold, 99th percentile normalized rate: {outlier_cutoff:.3f}")
print(outliers[["PRSCRBR_NPI", "Prscrbr_State_Abrvtn", "Prscrbr_Type", "credential_clean", "Tot_Clms", "Opioid_Tot_Clms", "opioid_rate_normalized"]].head(10))

la_provider = df.groupby("credential_clean", as_index=False).agg(mean_la_share=("la_opioid_share", "mean"), la_claims=("Opioid_LA_Tot_Clms", "sum"))
la_region = df.groupby("region", as_index=False).agg(mean_la_share=("la_opioid_share", "mean"), la_claims=("Opioid_LA_Tot_Clms", "sum"))
la_area = df.groupby("area_type", as_index=False).agg(mean_la_share=("la_opioid_share", "mean"), la_claims=("Opioid_LA_Tot_Clms", "sum"))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.barplot(data=la_provider, x="credential_clean", y="mean_la_share", ax=axes[0])
axes[0].set_title("LA Opioid Share by Credential")
sns.barplot(data=la_region.sort_values("mean_la_share", ascending=False), x="region", y="mean_la_share", ax=axes[1])
axes[1].set_title("LA Opioid Share by Region")
axes[1].tick_params(axis="x", rotation=25)
sns.barplot(data=la_area, x="area_type", y="mean_la_share", ax=axes[2])
axes[2].set_title("LA Opioid Share by Rural/Urban")
for ax in axes:
    ax.set_ylabel("LA opioid claims / opioid claims")
plt.tight_layout()
plt.show()

geo_drug_df = pd.read_csv(DATA_FILES["geo_drug"], low_memory=False)
geo_drug_df["drug_text"] = (geo_drug_df["Brnd_Name"].fillna("") + " " + geo_drug_df["Gnrc_Name"].fillna("")).str.lower()
oud_terms = r"buprenorphine|suboxone|sublocade|zubsolv|brixadi|buprenorphine hcl/naloxone"
pain_buprenorphine_terms = r"belbuca|butrans"
geo_drug_df["oud_buprenorphine_flag"] = geo_drug_df["drug_text"].str.contains(oud_terms, regex=True, na=False) & ~geo_drug_df["drug_text"].str.contains(pain_buprenorphine_terms, regex=True, na=False)
geo_drug_df["pain_opioid_flag"] = geo_drug_df["Opioid_Drug_Flag"].eq("Y") & ~geo_drug_df["oud_buprenorphine_flag"]

oud_vs_pain = pd.DataFrame({
    "category": ["OUD-related buprenorphine", "Pain-management opioids"],
    "total_claims": [
        geo_drug_df.loc[geo_drug_df["oud_buprenorphine_flag"], "Tot_Clms"].sum(),
        geo_drug_df.loc[geo_drug_df["pain_opioid_flag"], "Tot_Clms"].sum(),
    ],
    "unique_drugs": [
        geo_drug_df.loc[geo_drug_df["oud_buprenorphine_flag"], "Gnrc_Name"].nunique(),
        geo_drug_df.loc[geo_drug_df["pain_opioid_flag"], "Gnrc_Name"].nunique(),
    ],
})

plt.figure(figsize=(8, 5))
sns.barplot(data=oud_vs_pain, x="category", y="total_claims")
plt.title("OUD-Related Buprenorphine vs Pain-Management Opioid Claims")
plt.xlabel("")
plt.ylabel("Total claims")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

print("\nLA opioid provider comparison:")
print(la_provider)
print("\nBuprenorphine/OUD vs pain opioid comparison:")
print(oud_vs_pain)
print("\nInterpretation note: high buprenorphine rates can indicate greater OUD treatment access, not necessarily unsafe pain-opioid prescribing.")

## 04 - Machine Learning: High-Risk Prescribing Classification

Machine learning is used to screen for providers with elevated opioid prescribing risk, not to prove clinical wrongdoing. The target variable identifies providers in the top decile of normalized opioid prescribing rate among providers with at least 11 opioid claims. The models use provider credential, specialty, geography, rural/urban status, patient mix, claim volume, drug cost, and long-acting opioid share.

Model metrics should be interpreted together. Accuracy shows overall classification performance, precision shows how often predicted high-risk providers are actually high-risk, recall shows how many high-risk providers the model captures, F1 balances precision and recall, and MSE reports the average squared classification error for the binary target. The presentation-level comparison identifies Random Forest as the strongest model overall, with total claims, long-acting opioid share, specialty, opioid claims, and rural/urban status among the most important predictors. Geographic region and prescribing intensity contributed more strongly to model predictions than provider credential alone. The ML section supports risk screening and pattern detection, but conclusions should remain cautious and tied to public-health interpretation.


In [ ]:
# ============================================
# 04 - MACHINE LEARNING
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

ml_df = clean_provider_df.copy()
ml_df = ml_df.dropna(subset=["high_risk"]).copy()

feature_columns = [
    "credential_clean", "Prscrbr_State_Abrvtn", "region", "area_type", "Prscrbr_Type",
    "Prscrbr_RUCA", "Tot_Clms", "Tot_30day_Fills", "Tot_Drug_Cst", "Tot_Benes",
    "Bene_Avg_Age", "Bene_Avg_Risk_Scre", "la_opioid_share",
]
feature_columns = [col for col in feature_columns if col in ml_df.columns]

X = ml_df[feature_columns]
y = ml_df["high_risk"].astype(int)

sample_n = min(150_000, len(ml_df))
if len(ml_df) > sample_n:
    sampled = ml_df.sample(sample_n, random_state=42)
    X = sampled[feature_columns]
    y = sampled["high_risk"].astype(int)

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [col for col in X.columns if col not in categorical_features]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50))]), categorical_features),
    ]
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_leaf=100, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=12, min_samples_leaf=50, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

metrics = []
confusions = {}
fitted_models = {}
for model_name, model in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    fitted_models[model_name] = pipe
    confusions[model_name] = confusion_matrix(y_test, preds)
    metrics.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "mse": mean_squared_error(y_test, preds),
    })
    print(f"\n{model_name}")
    print(classification_report(y_test, preds, zero_division=0))

model_results = pd.DataFrame(metrics).sort_values("f1", ascending=False)
print("\nFormal model comparison:")
print(model_results)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (model_name, cm) in zip(axes, confusions.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(model_name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

best_tree_model = fitted_models["Random Forest"]
feature_names = best_tree_model.named_steps["preprocess"].get_feature_names_out()
importances = best_tree_model.named_steps["model"].feature_importances_
feature_importance = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 7))
sns.barplot(data=feature_importance, y="feature", x="importance")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()

print("\nTop feature importances:")
print(feature_importance)
print("\nLimitations: model performance reflects claims-file patterns and the engineered high-risk threshold; it does not prove clinical inappropriateness or patient-level harm.")

## 05 - Final Interpretation and Limitations

The final interpretation matches the presentation and poster narrative: opioid prescribing varies by geography and provider characteristics, normalized rates make those comparisons fairer, and model-based screening can help identify provider groups for follow-up review. The notebook supports that story with Washington-versus-region trends, rural/urban comparisons, provider credential summaries, specialty summaries, long-acting opioid comparisons, OUD/buprenorphine comparisons, and ML model results.

The strongest findings are: rural providers and some non-physician provider types show higher normalized opioid prescribing rates; long-acting opioids are less common overall but concentrate among a small subset of providers; Washington declined from 2017 to 2023 but remained higher than the aggregated 2023 regional weighted rates shown here; and model results can support targeted review, education, and monitoring rather than causal claims about wrongdoing.

The results are useful for population-level screening, but they do not prove clinical appropriateness or patient-level harm. Medicare Part D claims do not include all payers, the provider file is a 2023-only cross-section, some values are suppressed or missing, and claims data do not include full clinical context. Provider-per-capita normalization was not implemented because this notebook does not include state-level provider workforce denominators or beneficiary population denominators by provider group. This means the analysis compares rates within claims data, not opioid prescribing per resident or per provider population. Differences between preliminary and comparative analyses may reflect changes in normalization, weighting, and regional aggregation methodology.

Peer verification was not performed as part of this individual project. The workflow includes schema checks and reproducible code, but the final interpretations should still be reviewed by another analyst or classmate before submission. Peer review can catch wording issues, confirm that figures match claims, and reduce the risk of over-interpreting model results.

Key recommendations are to expand provider education on safe opioid prescribing, improve access to non-opioid pain management, increase OUD treatment availability in rural areas, use data-driven risk screening for quality improvement, and continue monitoring long-acting opioid use and outlier patterns.

All figures are designed to render cleanly inside the notebook and exported HTML. The code uses consistent figure sizes, clear titles, readable labels, and `tight_layout()` before display. External PNG export is intentionally not enabled so the project remains self-contained in `Project2.ipynb`.


In [ ]:
# ============================================
# 05 - FINAL OUTPUT SUMMARY
# ============================================

final_tables = {
    "provider_summary": provider_summary,
    "region_summary": region_summary,
    "area_summary": area_summary,
    "specialty_summary": specialty_summary,
    "washington_provider_summary": wa_provider_summary,
    "model_results": model_results,
    "oud_vs_pain": oud_vs_pain,
}

print("Final tables available in this notebook session:")
for name, table in final_tables.items():
    print(f"\n{name}: {table.shape[0]} rows x {table.shape[1]} columns")
    display(table.head())

print("\nNotebook conclusion:")
print("The completed workflow supports reproducible cleaning, Washington-vs-region trend analysis, normalized provider comparisons, LA opioid analysis, buprenorphine/OUD distinction, and model-based high-risk provider screening. All derived work is kept inside Project2.ipynb.")
